In [ ]:
import pandas as pd
from pathlib import Path
import os,sys
import numpy as np
from SpaceBalls.utils import nanrms

base_dir = '/media/monte_share/EEI_estimations'
base_dir_path = Path(base_dir)
out_dict = {'names': [], '1-yr_rms': [], '6-month_rms': [], '1-month_rms': []}

for folder in base_dir_path.iterdir():
    if folder.is_dir():
        out_dict["names"].append(folder)
        try:
            one_yr_rms = nanrms(np.load(os.path.join(folder, 'error_time_series_1-year.npy')))
            print(f"{folder.name}: 1-yr error RMS is {one_yr_rms} W/m^2")
        except:
            pass



In [ ]:
import sqlite3
import json
import hashlib

def make_key(input_dict):

    input_dict_filtererd = {key:value for (key, value) in input_dict.items() 
                            if (not(isinstance(value, dict)) and value is not None) or 
                            (isinstance(value, dict) and value)} # filter None entries and empty dicts
    key_string = json.dumps(input_dict_filtererd, sort_keys=True)
    
    #hash_str = hashlib.sha256(key_string.encode()).hexdigest()
    hash_str = hashlib.blake2b(key_string.encode(), digest_size=8).hexdigest()
    return hash_str


estimation_config_dict = {'EEI_truth_name': "EEI_truth_1", 
              'satellites': ['sc_A1', 'sc_A2', 'sc_A3'],
              'accelerometer_errors': {},
              'model_errors': {}}

estimation_key = make_key(estimation_config_dict)
acc_errors_key = make_key(estimation_config_dict['accelerometer_errors'])
model_errors_key = make_key(estimation_config_dict['model_errors'])

output_dict = {'1-month_avg_error': {'time_series': None,
                                     'rms_time_series': None},
                '6-month_avg_error': {'time_series': None,
                                     'rms_time_series': None},
                '1-year_avg_error': {'time_series': None,
                                     'rms_time_series': None}}



In [ ]:
import sqlite3

conn = sqlite3.connect('/media/monte_share/EEI_estimations/EEI_estimations_database.db')

c = conn.cursor()

c.execute("""CREATE TABLE IF NOT EXISTS config_table (
            case_hash text,
            EEI_truth_name text,
            satellites text,
            accelerometer_errors text,
            model_errors text
            )""")

c.execute("""CREATE TABLE IF NOT EXISTS EEI_postproc_results (
            case_hash text,

            )""")


def insert_emp(emp):
    with conn:
        c.execute("INSERT INTO employees VALUES (:first, :last, :pay)", {'first': emp.first, 'last': emp.last, 'pay': emp.pay})


def get_emps_by_name(lastname):
    c.execute("SELECT * FROM employees WHERE last=:last", {'last': lastname})
    return c.fetchall()


def update_pay(emp, pay):
    with conn:
        c.execute("""UPDATE employees SET pay = :pay
                    WHERE first = :first AND last = :last""",
                  {'first': emp.first, 'last': emp.last, 'pay': pay})


def remove_emp(emp):
    with conn:
        c.execute("DELETE from employees WHERE first = :first AND last = :last",
                  {'first': emp.first, 'last': emp.last})
        

conn.close()